# Multilingual Text Intelligence — Colab Demo

**NLP project:** Hindi–English code-mixed text processing and token-level language identification.

This notebook trains a lightweight **character n-gram + local-context classifier** on the public **COMI-LINGUA LID** dataset. Each token is classified as:

- `hi` — Hindi
- `en` — English
- `ot` — Other (punctuation and other linguistic units)

It evaluates on the dataset's held-out test split and launches a Gradio interface for live input.

> This is a baseline implementation for the project demonstration. It is not the 8B-parameter COMI-LINGUA model and does not claim to reproduce the paper's published results. Scores are computed by this notebook on the selected data subset.

In [ ]:
!pip -q install -U datasets scikit-learn gradio matplotlib "pandas==2.2.3" "bottleneck>=1.5"

In [ ]:
import ast
import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gradio as gr

from datasets import load_dataset
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

RANDOM_STATE = 42
MAX_TRAIN_SENTENCES = 10_000  # Reduce to 4_000 if your Colab session is slow.
MAX_TEST_SENTENCES = 2_000

LABELS = ["hi", "en", "ot"]
LABEL_NAMES = {
    "hi": "Hindi",
    "en": "English",
    "ot": "Other",
}

print("Libraries ready.")

## 1. Load the COMI-LINGUA language-identification split

COMI-LINGUA provides a LID configuration with token-wise Hindi, English and Other labels. The dataset card documents separate train and test splits and three annotator columns.

In [ ]:
DATASET_ID = "LingoIITGN/COMI-LINGUA"

dataset = load_dataset(DATASET_ID, "LID")

print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Test rows:", len(dataset["test"]))

# Preview the structure of one row before parsing annotations.
first_row = dataset["train"][0]
for column, value in first_row.items():
    print(f"\n{column}: {str(value)[:300]}")

## 2. Convert annotator data into token-level examples

The dataset contains three annotator columns. We use a majority vote for each token. If an annotator's token at a position does not match the reference token, that vote is skipped rather than forcing a potentially incorrect alignment.

Each training example contains the current token and its nearby tokens as context. This gives the baseline some context for ambiguous Romanized words.

In [ ]:
def parse_annotation(value):
    """Parse one annotation cell into [(token, label), ...]."""
    if value is None:
        return []

    obj = value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        parsed = None
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(value)
                break
            except (ValueError, SyntaxError, json.JSONDecodeError):
                continue
        if parsed is None:
            return []
        obj = parsed

    if isinstance(obj, np.ndarray):
        obj = obj.tolist()
    if not isinstance(obj, (list, tuple)):
        return []

    label_map = {
        "hi": "hi", "h": "hi", "hindi": "hi",
        "en": "en", "e": "en", "english": "en",
        "ot": "ot", "o": "ot", "other": "ot",
    }

    parsed_items = []
    for item in obj:
        if isinstance(item, dict):
            token = item.get("key", item.get("token", item.get("word")))
            label = item.get("value", item.get("label", item.get("lang")))
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            token, label = item[0], item[1]
        else:
            continue

        if token is None or label is None:
            continue

        token = str(token).strip()
        label = label_map.get(str(label).strip().lower())
        if token and label:
            parsed_items.append((token, label))

    return parsed_items


def find_annotation_columns(columns):
    """Find the three human annotator columns in the LID split."""
    cols = [c for c in columns if "annotator" in c.lower()]
    if not cols:
        cols = [
            c for c in columns
            if "annotated by" in c.lower()
        ]
    return cols[:3]


def context_string(tokens, index):
    """Build a local-context feature string for one token."""
    def safe(word):
        return str(word).lower().replace(" ", "_")

    current = safe(tokens[index])
    previous = safe(tokens[index - 1]) if index > 0 else "<START>"
    following = safe(tokens[index + 1]) if index + 1 < len(tokens) else "<END>"
    return f"word={current} previous={previous} next={following}"


def rows_to_examples(split, max_rows, annotation_columns, split_name):
    features, labels = [], []
    usable_rows = 0
    skipped_rows = 0

    # Shuffle with a fixed seed so the sampled subset is reproducible.
    split = split.shuffle(seed=RANDOM_STATE)
    total_rows = min(len(split), max_rows)

    for row_index in range(total_rows):
        row = split[row_index]
        annotations = [
            parse_annotation(row.get(column))
            for column in annotation_columns
        ]
        annotations = [annotation for annotation in annotations if annotation]

        if not annotations:
            skipped_rows += 1
            continue

        # Use the longest available annotation as the reference token sequence.
        reference = max(annotations, key=len)
        tokens = [token for token, _ in reference]
        row_added = 0

        for token_index, token in enumerate(tokens):
            votes = []
            for annotation in annotations:
                if token_index >= len(annotation):
                    continue
                other_token, other_label = annotation[token_index]
                if other_token.casefold() == token.casefold():
                    votes.append(other_label)

            if not votes:
                continue

            label = Counter(votes).most_common(1)[0][0]
            if label not in LABELS:
                continue

            features.append(context_string(tokens, token_index))
            labels.append(label)
            row_added += 1

        if row_added:
            usable_rows += 1
        else:
            skipped_rows += 1

    print(
        f"{split_name}: read {total_rows:,} rows; "
        f"usable sentences {usable_rows:,}; skipped {skipped_rows:,}"
    )
    print(f"{split_name}: {len(features):,} token examples | {Counter(labels)}")
    return features, labels


train_columns = find_annotation_columns(dataset["train"].column_names)
test_columns = find_annotation_columns(dataset["test"].column_names)

print("Training annotator columns:", train_columns)
print("Test annotator columns:", test_columns)

if not train_columns or not test_columns:
    raise ValueError(
        "Could not find annotator columns. Check the printed column names "
        "and update find_annotation_columns()."
    )

X_train_text, y_train = rows_to_examples(
    dataset["train"], MAX_TRAIN_SENTENCES, train_columns, "Train"
)
X_test_text, y_test = rows_to_examples(
    dataset["test"], MAX_TEST_SENTENCES, test_columns, "Test"
)

if not X_train_text or not X_test_text:
    raise ValueError(
        "No token examples were extracted. Inspect the annotation preview above."
    )

## 3. Train the lightweight NLP baseline

We convert the current token and its local context into character TF-IDF n-gram features. An SGD logistic classifier learns to predict Hindi, English or Other.

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

classifier = SGDClassifier(
    loss="log_loss",
    max_iter=20,
    tol=1e-3,
    alpha=1e-5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

classifier.fit(X_train, y_train)

print("Training complete.")
print("Model classes:", classifier.classes_)
print("Training token examples:", X_train.shape[0])

## 4. Evaluate on the held-out test split

The following metrics are calculated from this run on the selected test subset. They are not copied from the research paper.

In [ ]:
y_pred = classifier.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Macro F1 on selected held-out tokens: {macro_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=LABELS,
        target_names=[LABEL_NAMES[label] for label in LABELS],
        zero_division=0,
    )
)

cm = confusion_matrix(y_test, y_pred, labels=LABELS)
display_plot = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[LABEL_NAMES[label] for label in LABELS],
)
display_plot.plot(cmap="Blues", values_format="d")
plt.title("Token-level language identification")
plt.show()

## 5. Predict labels for a new Hinglish sentence

The input is split into words and punctuation. For each token, the model sees the current token and its neighbouring tokens.

In [ ]:
TOKEN_PATTERN = re.compile(
    r"[#@]?\w+(?:['’]\w+)*|[^\w\s]",
    flags=re.UNICODE,
)


def predict_tokens(text):
    tokens = TOKEN_PATTERN.findall(text.strip())
    if not tokens:
        return []

    contexts = [
        context_string(tokens, index)
        for index in range(len(tokens))
    ]
    features = vectorizer.transform(contexts)
    predictions = classifier.predict(features)
    scores = classifier.predict_proba(features)

    results = []
    for token, label, score_row in zip(tokens, predictions, scores):
        results.append({
            "Token": token,
            "Language": LABEL_NAMES[label],
            "Label": label,
            "Model score": round(float(np.max(score_row)), 3),
        })
    return results


def analyse_text(text):
    rows = predict_tokens(text)
    if not rows:
        empty = pd.DataFrame(
            columns=["Token", "Language", "Label", "Model score"]
        )
        return empty, "Please enter some text."

    counts = Counter(
        row["Label"]
        for row in rows
        if row["Label"] in ("hi", "en", "ot")
    )
    language_total = counts.get("hi", 0) + counts.get("en", 0)
    hindi_pct = counts.get("hi", 0) / language_total * 100 if language_total else 0
    english_pct = counts.get("en", 0) / language_total * 100 if language_total else 0

    active = [
        label for label in ("hi", "en")
        if counts.get(label, 0) > 0
    ]
    if len(active) == 2:
        sentence_type = "Hindi-English code-mixed text"
    elif active:
        sentence_type = f"Mostly {LABEL_NAMES[active[0]]}"
    else:
        sentence_type = "No Hindi or English tokens identified"

    switches = []
    for index in range(len(rows) - 1):
        left = rows[index]["Label"]
        right = rows[index + 1]["Label"]
        if left in ("hi", "en") and right in ("hi", "en") and left != right:
            switches.append(
                f"{rows[index]['Token']} → {rows[index + 1]['Token']}"
            )

    summary = (
        f"**Sentence analysis:** {sentence_type}\n\n"
        f"**Hindi:** {counts.get('hi', 0)} tokens ({hindi_pct:.1f}%)  |  "
        f"**English:** {counts.get('en', 0)} tokens ({english_pct:.1f}%)  |  "
        f"**Other:** {counts.get('ot', 0)} tokens\n\n"
        f"**Language switches:** {', '.join(switches) if switches else 'No adjacent Hindi-English switch detected'}\n\n"
        f"*Model scores are classifier scores and are not guaranteed to be calibrated probabilities.*"
    )

    return pd.DataFrame(rows), summary


example = (
    "I will kal office jaa raha hun, "
    "aur meeting ke baad report email kar dunga."
)
example_table, example_summary = analyse_text(example)

print(example)
display(example_table)
print(example_summary)

## 6. Launch the live demo for your faculty

Run the next cell. Colab will display a temporary Gradio link. Open it, enter a Hindi, English or Hinglish sentence, and click **Analyze text**.

In [ ]:
def gradio_analyse(text):
    table, summary = analyse_text(text)
    return table, summary


with gr.Blocks(title="Multilingual Text Intelligence — NLP Demo") as demo:
    gr.Markdown("""
    # Multilingual Text Intelligence
    ### Hindi-English Code-Mixed Text Processing and Language Identification

    Enter a sentence in Hindi, English or Hinglish. The model labels each token.
    """)

    input_text = gr.Textbox(
        label="Enter text",
        lines=3,
        value=(
            "I will kal office jaa raha hun, "
            "aur meeting ke baad report email kar dunga."
        ),
    )
    analyse_button = gr.Button("Analyze text", variant="primary")
    output_table = gr.Dataframe(
        label="Token-level language labels",
        interactive=False,
    )
    output_summary = gr.Markdown(label="Sentence summary")

    analyse_button.click(
        gradio_analyse,
        inputs=input_text,
        outputs=[output_table, output_summary],
    )

# share=True creates a temporary public URL for the demo session.
demo.launch(share=True, debug=False)

## What to explain during the demonstration

1. The dataset contains token-level Hindi, English and Other annotations from three annotators.
2. We create majority-vote labels and build a local-context feature for each token.
3. Character n-gram TF-IDF features are used by a lightweight SGD classifier.
4. The model predicts one language label per token and produces a sentence-level summary.
5. We evaluate on a held-out test split using macro-F1 and a confusion matrix.

**Limitations:** Romanized Hindi can be ambiguous. This baseline uses only a small local context window and will make mistakes on unfamiliar spellings, names and borrowed words. It is the first working baseline, not the final transformer comparison.

### Research reference

Sheth, Rajvee, Himanshu Beniwal, and Mayank Singh. 2025. *COMI-LINGUA: Expert Annotated Large-Scale Dataset for Multitask NLP in Hindi-English Code-Mixing*. Findings of EMNLP 2025.

Paper: https://aclanthology.org/2025.findings-emnlp.422/

Dataset: https://huggingface.co/datasets/LingoIITGN/COMI-LINGUA

Dataset license: CC BY 4.0.